# 01 — Class 27: Compute & Runtime Lab

Free Edition is serverless-only, so this notebook practices the **real execution behavior** you can inspect now:
- Spark version/config,
- narrow operation,
- shuffle,
- physical plan,
- skew.

Classic Standard/Dedicated decisions are taught in the guide.


In [0]:
import sys
import os
from pathlib import Path

marker = "RETAIL_LAB.json"
cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

print(cwd)

for candidate in [cwd, *cwd.parents]:
    # print(f"Looking for {marker} in {candidate}")
    if (candidate / marker).exists():
        #print("Project root :", candidate)
        PROJECT_ROOT = str(candidate)
        #print("Project root :", PROJECT_ROOT)
        break


if PROJECT_ROOT is None:
    for search_root in [Path("/Workspace/Users"), Path("/Workspace/Shared"), Path("/Workspace")]:
        if not search_root.exists():
            continue
        for current_root, dirs, files in os.walk(search_root):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"Trash", ".Trash"}]
            if marker in files:
                PROJECT_ROOT = current_root
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project not found. Import the complete ZIP into Databricks Workspace."
    )

SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from bootstrap import ensure_lab

ctx = ensure_lab(
    spark=spark,
    project_root=PROJECT_ROOT,
)

print("Project root :", PROJECT_ROOT)
print("Catalog      :", ctx.catalog)
print("Schema       :", ctx.schema)
print("Volume       :", ctx.volume)
print("Source root  :", ctx.source_root)


In [0]:
from pyspark.sql import functions as F

print("Spark version:", spark.version)

for key in ["spark.sql.adaptive.enabled", "spark.sql.shuffle.partitions"]:
    try:
        print(key, "=", spark.conf.get(key))
    except Exception as exc:
        print(key, "= unavailable:", exc)


In [0]:
orders_df = (
    spark.read.option("header", True)
    .csv(f"{ctx.source_root}/orders.csv")
)
skew_sales_df = (
    spark.read.option("header", True)
    .csv(f"{ctx.source_root}/skew_sales.csv")
)

assert orders_df.count() == 126
assert skew_sales_df.count() == 1000


In [0]:
delivered_orders_df = (
    orders_df
    .filter(F.col("order_status") == "DELIVERED")
    .select("order_id", "customer_id", "shipping_city")
)

delivered_orders_df.show()
# delivered_orders_df.explain("formatted")
# assert delivered_orders_df.count() == 78


In [0]:
city_orders_df = (
    orders_df
    .groupBy("shipping_city")
    .count()
    .orderBy(F.col("count").desc())
)

city_orders_df.explain("formatted")
display(city_orders_df)


In [0]:
city_skew_df = (
    skew_sales_df
    .groupBy("city")
    .count()
    .orderBy(F.col("count").desc())
)

display(city_skew_df)

# top = city_skew_df.first()
# assert top["city"] == "Hyderabad"
# assert top["count"] == 700


### Compute-selection recap

```text
Interactive Python/SQL → Serverless first
Automated ETL         → Serverless jobs first
SQL/BI                → SQL warehouse
RDD / R / GPU         → Classic Dedicated
Custom classic config → Classic jobs compute
```
